# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.
%load_ext dotenv
%dotenv 

import pandas as pd
import numpy as np



In [2]:
import yfinance as yf
import os
import sys
sys.path.append(os.getenv('SRC_DIR'))

import dask.dataframe as dd

c:\Users\nnp\miniconda3\envs\dsi_participant\lib\site-packages\dask\dataframe\_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 11.0.0. Please consider upgrading.
  warnings.warn(


+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [3]:
from glob import glob

# Write your code below.

#Load the environment variable 'PRICE_DATA':

price_file = os.getenv("PRICE_DATA")

price_file


'../../05_src/data/prices/'

In [4]:
#Use glob to find path of parquet files:
price_glob = glob(price_file + "**/*.parquet", recursive=True)
df = dd.read_parquet(price_glob)
df


,Date,Ticker,Adj Close,Close,High,Low,Open,Volume,Year
npartitions=3328,,,,,,,,,
,"datetime64[ns, UTC]",object,float64,float64,float64,float64,float64,float64,int32
,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [5]:
# Write your code below.

dd_shift = df.groupby('Ticker', group_keys=False).apply(
   lambda x: x.assign(Close_lag_1 = x['Close'].shift(1))
)
dd_shift


C:\Users\nnp\AppData\Local\Temp\ipykernel_16604\2144119371.py:3: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_shift = df.groupby('Ticker', group_keys=False).apply(


,Date,Ticker,Adj Close,Close,High,Low,Open,Volume,Year,Close_lag_1
npartitions=3328,,,,,,,,,,
,"datetime64[ns, UTC]",object,float64,float64,float64,float64,float64,float64,int32,float64
,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...


In [6]:

dd_feat = dd_shift.assign (
    Adj_Close_lag_1 = dd_shift.groupby('Ticker', group_keys=False)['Adj Close'].shift(1),
    returns = (dd_shift['Close']/dd_shift['Close_lag_1']) - 1,
    hi_lo_range = dd_shift['High']-dd_shift['Low']
)

C:\Users\nnp\AppData\Local\Temp\ipykernel_16604\524462402.py:2: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  Adj_Close_lag_1 = dd_shift.groupby('Ticker', group_keys=False)['Adj Close'].shift(1),


In [7]:
dd_feat

,Date,Ticker,Adj Close,Close,High,Low,Open,Volume,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range
npartitions=3328,,,,,,,,,,,,,
,"datetime64[ns, UTC]",object,float64,float64,float64,float64,float64,float64,int32,float64,float64,float64,float64
,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [8]:
# Write your code below.
df_panda = dd_feat.compute()

df_panda

,Date,Ticker,Adj Close,Close,High,Low,Open,Volume,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range
16016,2000-12-28 00:00:00+00:00,CTSH,1.299723,1.442708,1.447917,1.273438,1.286458,2260800.0,2000,1.286458,1.158959,0.121458,0.174479
268240,2016-08-30 00:00:00+00:00,CTSH,52.170628,57.910000,58.189999,57.509998,58.090000,2430100.0,2016,57.849998,52.116577,0.001037,0.680000
9404,2000-08-01 00:00:00+00:00,TYL,2.250000,2.250000,2.500000,2.187500,2.500000,92400.0,2000,2.562500,2.562500,-0.121951,0.312500
326323,2020-04-08 00:00:00+00:00,ROP,309.705902,318.079987,320.760010,306.910004,311.750000,482600.0,2020,308.109985,299.998352,0.032359,13.850006
130995,2008-02-25 00:00:00+00:00,ROP,52.951096,58.580002,58.889999,56.299999,57.619999,1254100.0,2008,59.470001,53.755581,-0.014966,2.590000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
152581,2009-06-26 00:00:00+00:00,AKAM,20.090000,20.090000,20.139999,19.570000,19.940001,5495100.0,2009,20.110001,20.110001,-0.000995,0.570000
54405,2003-05-23 00:00:00+00:00,AKAM,3.640000,3.640000,3.750000,3.600000,3.750000,773500.0,2003,3.640000,3.640000,0.000000,0.150000
220219,2013-09-06 00:00:00+00:00,TXN,28.914162,39.180000,39.599998,38.980000,39.599998,4770400.0,2013,39.480000,29.135548,-0.007599,0.619999
33851,2002-02-12 00:00:00+00:00,TXN,20.225828,31.820000,32.400002,31.129999,31.700001,9444400.0,2002,31.950001,20.308458,-0.004069,1.270002


In [10]:
df_panda['rolling_means'] = df_panda['returns'].rolling(10).mean()
df_panda

,Date,Ticker,Adj Close,Close,High,Low,Open,Volume,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range,rolling_means
16016,2000-12-28 00:00:00+00:00,CTSH,1.299723,1.442708,1.447917,1.273438,1.286458,2260800.0,2000,1.286458,1.158959,0.121458,0.174479,NaN
268240,2016-08-30 00:00:00+00:00,CTSH,52.170628,57.910000,58.189999,57.509998,58.090000,2430100.0,2016,57.849998,52.116577,0.001037,0.680000,NaN
9404,2000-08-01 00:00:00+00:00,TYL,2.250000,2.250000,2.500000,2.187500,2.500000,92400.0,2000,2.562500,2.562500,-0.121951,0.312500,NaN
326323,2020-04-08 00:00:00+00:00,ROP,309.705902,318.079987,320.760010,306.910004,311.750000,482600.0,2020,308.109985,299.998352,0.032359,13.850006,NaN
130995,2008-02-25 00:00:00+00:00,ROP,52.951096,58.580002,58.889999,56.299999,57.619999,1254100.0,2008,59.470001,53.755581,-0.014966,2.590000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
152581,2009-06-26 00:00:00+00:00,AKAM,20.090000,20.090000,20.139999,19.570000,19.940001,5495100.0,2009,20.110001,20.110001,-0.000995,0.570000,-0.010608
54405,2003-05-23 00:00:00+00:00,AKAM,3.640000,3.640000,3.750000,3.600000,3.750000,773500.0,2003,3.640000,3.640000,0.000000,0.150000,-0.006345
220219,2013-09-06 00:00:00+00:00,TXN,28.914162,39.180000,39.599998,38.980000,39.599998,4770400.0,2013,39.480000,29.135548,-0.007599,0.619999,-0.003861
33851,2002-02-12 00:00:00+00:00,TXN,20.225828,31.820000,32.400002,31.129999,31.700001,9444400.0,2002,31.950001,20.308458,-0.004069,1.270002,-0.001359


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

<br>
<br>
No, it was not necessary to convert to pandas to compute rolling means. We can do that directly using Dask. Coverting to pandas requires loading all the data into memory which takes a lot of time especially with such large datasets. Using Dask allows us to utilize its parallel processing abilities which significantly reduces computational time and memory requirements.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.